In [15]:
import numpy as np
import pandas as pd
import xarray as xr

This notebook calculates skill scores for remote estimates of evapotranspiration (ET).

Data are assumed to be in the directory ../DATA
- The list of stations to be analysed, identified by two letter acronyms, is in ../DATA/Stations.txt
- The data for each station is in a subfolder named by its acronym, e.g. ../DATA/JC

Read in the list of station acronyms.

In [16]:
STNS = np.loadtxt("../DATA/Stations.txt",dtype='U2')

Read in the list of ET product acronyms.
- PM should always be the first in this list.

In [17]:
VNS = np.loadtxt("../DATA/Products.txt",dtype='<U10')

Read in data for each station

In [18]:
i=0
STN = STNS[i]
VN = VNS[0]
qwe = xr.open_dataarray("../DATA/"+STN+"/"+VN+"_"+STN+".nc")
DateTime = pd.to_datetime(np.ravel( qwe['Year']*1000 + qwe['DOY']), format='%Y%j')
Values = np.ravel(qwe, order='F').reshape(1, 1, 276)
STN_DATA = xr.DataArray(data=Values,
                        coords=[[STN], [VN], DateTime],
                        dims=["Station", "Product", "datetime"],
                        name = "ET")
for j in range(VNS.size-1):
    VN = VNS[j+1]
    qwe = xr.open_dataarray("../DATA/"+STN+"/"+VN+"_"+STN+".nc")
    DateTime = pd.to_datetime(np.ravel( qwe['Year']*1000 + qwe['DOY']), format='%Y%j')
    Values = np.ravel(qwe, order='F').reshape(1, 1, 276)
    qwe_DATA = xr.DataArray(data=Values,
                            coords=[[STN], [VN], DateTime],
                            dims=["Station", "Product", "datetime"],
                            name = "ET")
    STN_DATA = xr.merge([STN_DATA, qwe_DATA], join="outer", compat="no_conflicts")

In [19]:
for i in range(STNS.size-1):
    STN = STNS[i+1]
    for j in range(VNS.size):
        VN = VNS[j]
        qwe = xr.open_dataarray("../DATA/"+STN+"/"+VN+"_"+STN+".nc")
        DateTime = pd.to_datetime(np.ravel( qwe['Year']*1000 + qwe['DOY']), format='%Y%j')
        Values = np.ravel(qwe, order='F').reshape(1, 1, 276)
        qwe_DATA = xr.DataArray(data=Values,
                            coords=[[STN], [VN], DateTime],
                            dims=["Station", "Product", "datetime"],
                           name = "ET")
        STN_DATA = xr.merge([STN_DATA, qwe_DATA], join="outer", compat="no_conflicts")

Calculate Errors and Skill Scores

In [20]:
STN_ERR = STN_DATA['ET'] - STN_DATA['ET'].loc[dict(Product='PM')]
STN_BIAS = STN_ERR.mean(dim="datetime")
STN_RMSE = np.sqrt((STN_ERR**2).mean(dim="datetime"))
STN_CC = xr.corr(STN_DATA['ET'].loc[dict(Product='PM')], STN_DATA['ET'].loc[dict(Product=VNS[:])], dim="datetime" )

Save data files

In [21]:
STN_DATA.to_netcdf("../DATA/STN_DATA.nc")
STN_ERR.to_netcdf("../DATA/STN_ERR.nc")
STN_BIAS.to_netcdf("../DATA/STN_BIAS.nc")
STN_RMSE.to_netcdf("../DATA/STN_RMSE.nc")
STN_CC.to_netcdf("../DATA/STN_CC.nc")